In [8]:
import pandas as pd

# Cargar el dataset generado por el pipeline
df = pd.read_csv("include/output/movies_2026-09-21.csv")
df.head(5)


,movie_id,title,release_year,release_month,release_day_of_week,budget,revenue,runtime,original_language,primary_genre,secondary_genre,primary_production_company,co_production_company,director_popularity,lead_actor_popularity,co_star_popularity,vote_average,vote_count
0,969681,Spider-Man: Brand New Day,2026,7,2,225000000.0,2.478502e+09,145,en,Science Fiction,Action,Marvel Studios,Columbia Pictures,2.5091,12.9617,12.1356,7.86,2798
1,1423191,Resident Evil,2026,9,2,75000000.0,1.083000e+08,95,en,Horror,Science Fiction,Constantin Film,Subconscious,7.5087,12.0879,4.9997,7.30,241
2,1101383,The End of Oak Street,2026,8,2,80000000.0,1.203637e+08,100,en,Science Fiction,Mystery,Warner Bros. Pictures,Domain Entertainment,2.2613,16.9593,6.9073,7.00,1337
3,1204680,Coyote vs. Acme,2026,8,3,72000000.0,7.982511e+07,103,en,Comedy,Adventure,Troll Court Entertainment,Keylight Pictures,0.8813,4.1069,3.1287,7.60,468
4,1368337,The Odyssey,2026,7,2,250000000.0,1.721315e+09,173,en,Adventure,Action,Universal Pictures,Syncopy,10.8536,9.8324,12.9617,8.00,3824


In [9]:
# Debe dar True
df["movie_id"].is_unique

True

In [10]:
# Piden > 1.000 filas
len(df)

10000

In [ ]:
# Piden >= 5 columnas útiles (tenemos 18)
df.shape

(10000, 18)

In [12]:
# Verificación de numéricas, categóricas y temporales
df.dtypes.value_counts()

int64      6
str        6
float64    6
Name: count, dtype: int64

In [13]:
# Proporción de nulos por columna ordenada
df.isna().mean().sort_values(ascending=False)

movie_id                      0.0
title                         0.0
release_year                  0.0
release_month                 0.0
release_day_of_week           0.0
budget                        0.0
revenue                       0.0
runtime                       0.0
original_language             0.0
primary_genre                 0.0
secondary_genre               0.0
primary_production_company    0.0
co_production_company         0.0
director_popularity           0.0
lead_actor_popularity         0.0
co_star_popularity            0.0
vote_average                  0.0
vote_count                    0.0
dtype: float64

In [14]:
# Debe dar una lista vacía []
df.columns[df.isna().all()].tolist()

[]

In [7]:
import pandas as pd
from pathlib import Path

# Buscamos el último CSV generado en include/output o include/frozen
csv_candidates = list(Path("include/output").glob("movies_*.csv")) + [
    Path("include/frozen/ultimo_ok.csv"),
    Path("include/frozen/tmdb_snapshot.csv"),
]

csv_path = next((p for p in csv_candidates if p.exists()), None)

if not csv_path:
    raise FileNotFoundError("No se encontró ningún CSV para analizar.")

df = pd.read_csv(csv_path, low_memory=False)
total = len(df)

# En TMDb, los valores ausentes de budget y revenue se registran típicamente como 0 o NaN
budget_valido = df["budget"].notna() & (df["budget"] > 0)
revenue_valido = (
    df["revenue"].notna() & (df["revenue"] > 0)
    if "revenue" in df.columns
    else pd.Series([False] * total)
)

print(f"Dataset analizado: {csv_path} (Total: {total} filas)\n")

print(f"--- BUDGET ---")
print(f"Con presupuesto válido (> 0): {budget_valido.sum()} ({budget_valido.mean():.1%})")
print(f"Con presupuesto en 0 o NaN:   {(~budget_valido).sum()} ({(~budget_valido).mean():.1%})\n")

if "revenue" in df.columns:
    print(f"--- REVENUE ---")
    print(f"Con recaudación válida (> 0): {revenue_valido.sum()} ({revenue_valido.mean():.1%})")
    print(f"Con recaudación en 0 o NaN:   {(~revenue_valido).sum()} ({(~revenue_valido).mean():.1%})\n")

    ambos_validos = budget_valido & revenue_valido
    print(f"--- AMBOS (Cálculo de ROI posible) ---")
    print(f"Películas con Budget > 0 y Revenue > 0: {ambos_validos.sum()} ({ambos_validos.mean():.1%})")
    print(f"Películas sin datos financieros completos: {(~ambos_validos).sum()} ({(~ambos_validos).mean():.1%})")
else:
    print("Nota: La columna 'revenue' todavía no está en tu CSV actual, pero la columna 'budget' ya te sirve de termómetro.")

Dataset analizado: include\output\movies_2026-09-21.csv (Total: 10000 filas)

--- BUDGET ---
Con presupuesto válido (> 0): 7178 (71.8%)
Con presupuesto en 0 o NaN:   2822 (28.2%)

--- REVENUE ---
Con recaudación válida (> 0): 7703 (77.0%)
Con recaudación en 0 o NaN:   2297 (23.0%)

--- AMBOS (Cálculo de ROI posible) ---
Películas con Budget > 0 y Revenue > 0: 6595 (66.0%)
Películas sin datos financieros completos: 3405 (34.1%)


In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("include/output/movies_2026-09-21.csv")

# 1. Perfil básico
print("=== PERFIL DEL DATASET ===")
print("Dimensiones (shape):", df.shape)
print("Clave única (movie_id):", df["movie_id"].is_unique)
print("\nTipos de datos:\n", df.dtypes.value_counts())

# 2. Distribución de ceros/ausentes en variables financieras
sin_presupuesto = (df["budget"] == 0).sum()
sin_revenue = (df["revenue"] == 0).sum()
completas_roi = ((df["budget"] > 0) & (df["revenue"] > 0)).sum()

print("\n=== COBERTURA FINANCIERA ===")
print(f"Películas con budget == 0: {sin_presupuesto} ({sin_presupuesto/len(df):.1%})")
print(f"Películas con revenue == 0: {sin_revenue} ({sin_revenue/len(df):.1%})")
print(f"Subconjunto con ROI calculable: {completas_roi} ({completas_roi/len(df):.1%})")

# 3. Asimetría de variables numéricas (skewness)
print("\n=== ASIMETRÍA (SKEW) DE NUMÉRICAS ===")
numeric_cols = df.select_dtypes(include=["number"]).columns
skew_series = df[numeric_cols].skew().sort_values(ascending=False)
print(skew_series)

# 4. Columnas constantes
const_cols = [col for col in df.columns if df[col].nunique() <= 1]
print("\nColumnas sin información (constantes):", const_cols)

=== PERFIL DEL DATASET ===
Dimensiones (shape): (10000, 18)
Clave única (movie_id): True

Tipos de datos:
 int64      6
str        6
float64    6
Name: count, dtype: int64

=== COBERTURA FINANCIERA ===
Películas con budget == 0: 2822 (28.2%)
Películas con revenue == 0: 2297 (23.0%)
Subconjunto con ROI calculable: 6595 (66.0%)

=== ASIMETRÍA (SKEW) DE NUMÉRICAS ===
revenue                  5.733395
vote_count               3.922537
budget                   3.642398
director_popularity      3.610738
co_star_popularity       2.254894
lead_actor_popularity    1.947572
movie_id                 1.637853
runtime                  1.270348
release_month           -0.133530
vote_average            -0.196138
release_day_of_week     -0.408891
release_year            -1.426162
dtype: float64

Columnas sin información (constantes): []
